# Activity 4 — Search and Load Sentinel-2

**Goal:** Reproduce the source notebook's STAC search and xarray loading stage.

> 🟢 **Follow Me** — run and understand a working example.  
> 🟡 **Your Turn** — complete or modify part of the code.  
> 🔵 **Challenge** — work out the solution from earlier examples.  
> 🔎 **Debug It** — diagnose an error or unexpected result.
>
> This version follows the uploaded **Fiji_cordia_2.0_prediction_Vanuatu** workflow, while breaking complex lines into beginner-friendly steps.

In [ ]:
from pystac_client import Client
from dask.distributed import Client as DaskClient
from odc.stac import load, configure_s3_access
import numpy as np
import xarray as xr
import odc.geo.xr  # noqa: F401

## 4.1 🟢 Follow Me — Connect to Earth Search

In [ ]:
catalog = "https://earth-search.aws.element84.com/v1"
client = Client.open(catalog)

## 4.2 🟡 Your Turn — Time period
The source workflow recommends experimenting with flowering months and cloud availability.

In [ ]:
datetime = ""   # e.g. "2018-03/2018-07"

## 4.3 🟢 Follow Me — Configure Dask/S3

In [ ]:
dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit='16GB')
configure_s3_access(cloud_defaults=True, requester_pays=True)

## 4.4 🟡 Your Turn — Search
Complete the collection and cloud threshold used by the source workflow.

In [ ]:
items = client.search(
    collections=["____________________"],
    bbox=bbox,
    datetime=datetime,
    query={"eo:cloud_cover": {"lt": ___}},
).item_collection()

print(f"Found {len(items)} items")

## 4.5 🟡 Your Turn — Load the exact bands
Complete `nir08`, `swir16`, and `scl`.

In [ ]:
data = load(
    items,
    measurements=[
        "red", "green", "blue",
        "_____", "_____", "_____"
    ],
    bbox=bbox,
    chunks={"x": 2048, "y": 2048},
    groupby="solar_day",
)

data

## 4.6 🟢 Follow Me — See what Sentinel-2 dates are available

Before masking clouds, let's first look at the imagery **through time**.

This helps us understand:

- which months have the clearest imagery,
- which months are heavily affected by clouds,
- whether vegetation appearance changes through the year,
- and which period may best match the flowering season of the target invasive species.

In [ ]:
print("Number of Sentinel-2 observations:", data.sizes["time"])

print("\nDates available:")
for date in data.time.values:
    print(str(date)[:10])

## 4.7 🟡 Your Turn — View one Sentinel-2 acquisition

Start with `time_index = 0`.

Then try `1`, `2`, `3`, and so on to compare different dates.

In [ ]:
# YOUR TURN
time_index = 0

data[["red", "green", "blue"]].isel(time=time_index).odc.explore(
    vmin=0,
    vmax=3000
)

## 4.8 🟢 Follow Me — Create monthly RGB composites

We can group all available Sentinel-2 observations by **calendar month** and calculate a median image for each month.

At this point, the data have **not been cloud masked yet**. That is intentional — we want to see the condition of the raw monthly imagery first.

In [ ]:
if data.sizes["time"] > 1:
    monthly_raw = data.groupby("time.month").median("time").compute()
    print("Months available:", monthly_raw.month.values)
else:
    monthly_raw = data.median("time").compute()
    print("Only one Sentinel-2 observation is available.")

## 4.9 🟡 Your Turn — View a selected month

Choose one of the month numbers printed above.

For example:

- `3` = March
- `4` = April
- `5` = May
- `6` = June

In [ ]:
# YOUR TURN
selected_month = 3

if "month" in monthly_raw.dims:
    monthly_raw[["red", "green", "blue"]].sel(
        month=selected_month
    ).odc.explore(
        vmin=0,
        vmax=3000
    )
else:
    monthly_raw[["red", "green", "blue"]].odc.explore(
        vmin=0,
        vmax=3000
    )

## 4.10 🔵 Challenge — Compare the months

Explore at least **three months** and record what you notice.

| Month | Cloud condition | Vegetation appearance | Potentially useful for detection? |
|---|---|---|---|
| | | | |
| | | | |
| | | | |

Discuss:

1. Which month looks clearest?
2. Which month has the most cloud?
3. Do canopy colours or vegetation patterns change?
4. Which month might be most useful during the target species' flowering period?

## 4.6 🔵 Challenge
Inspect `data.dims`, `data.data_vars`, and the number of time steps. Explain what `time`, `x`, and `y` represent.

In [ ]:
# CHALLENGE